In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import pandas as pd
from tqdm import tqdm_notebook

## Load Datasets

In [ ]:
data_dir = "data/AAAI2023Competition/"

In [ ]:
df_train_valid = pd.read_csv(os.path.join(data_dir,"train_valid_sequences.csv"))

In [ ]:
df_train_valid.head(1)

In [ ]:
valid_fold = 0 #use the 0 fold to train and valid
df_train = df_train_valid[df_train_valid['fold']!=valid_fold].copy()#train dataset for fold 0
df_valid = df_train_valid[df_train_valid['fold']==valid_fold].copy()#valid dataset for fold 0

In [ ]:
df_train.shape,df_valid.shape

In [ ]:
def flatten_dataset(df):
    interaction_list = []
    for _, row in tqdm_notebook(df.iterrows()):
        uid = row['uid']
        for question, concept, response in zip(row['questions'].split(","),
                                               row['concepts'].split(","),
                                               row['responses'].split(",")):
            if response == "-1":#remove the padding
                break
            interaction = {"uid": int(uid),
                           "question": int(question),
                           "concept": int(concept),
                           "response": int(response)}
            interaction_list.append(interaction)
    df_interaction = pd.DataFrame(interaction_list)
    print(f"# interaction is {len(interaction_list)}")
    return df_interaction

In [ ]:
df_train_inter = flatten_dataset(df_train)
df_valid_inter = flatten_dataset(df_valid)

## Train a Model

In [ ]:
df_train_inter.head(1)

### Question Level

In [ ]:
top_answers_que_level = df_train_inter.groupby(
    'question')['response'].agg("mean")
top_answers_dict_que_level = top_answers_que_level.to_dict()

### Concept Level

In [ ]:
top_answers_concept_level = df_train_inter.groupby('concept')[
    'response'].agg("mean")
top_answers_dict_concept_level = top_answers_concept_level.to_dict()

## Predict on the Validation Dataset

In [ ]:
df_valid_inter['que_pred'] = df_valid_inter['question'].apply(
    lambda x: top_answers_dict_que_level.get(x, 0))
df_valid_inter['concept_pred'] = df_valid_inter['concept'].apply(
    lambda x: top_answers_dict_concept_level.get(x, 0))

## Evaluate on the Validation Dataset

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
que_auc = roc_auc_score(df_valid_inter['response'],df_valid_inter['que_pred'])
que_auc

In [ ]:
concept_auc = roc_auc_score(df_valid_inter['response'],df_valid_inter['concept_pred'])
concept_auc

## Train use the Whole Dataset

In [ ]:
top_answers_que_level = pd.concat([df_train_inter,df_valid_inter]).groupby(
    'question')['response'].agg("mean")
top_answers_dict_que_level = top_answers_que_level.to_dict()

In [ ]:
# check again
df_valid_inter['que_pred'] = df_valid_inter['question'].apply(
    lambda x: top_answers_dict_que_level.get(x, 0))
que_auc = roc_auc_score(df_valid_inter['response'],df_valid_inter['que_pred'])
que_auc

## Predict on the Test Dataset

From the experiement, we notice the question level model has better performance, we use the question level model as our final model

In [ ]:
df_test = pd.read_csv(os.path.join(data_dir,"pykt_test.csv"))

In [ ]:
df_test

In [ ]:
predict_str_list = []
num_test = 0
for _, row in df_test.iterrows():
    predict_results = []
    for question, response, is_repeat in zip(row['questions'].split(","), 
                                             row['responses'].split(","), 
                                             row['is_repeat'].split(",")):
        question, response,is_repeat = int(question), int(response),int(is_repeat)
        if is_repeat!=0:#skip the repeat
            continue
        if response == -1:
            num_test += 1
            predict_results.append(top_answers_dict_que_level.get(question, 0))
    predict_str = ",".join([str(x) for x in predict_results])
    predict_str_list.append(predict_str)

In [ ]:
num_test

In [ ]:
df_submit = pd.DataFrame({"responses":predict_str_list})

In [ ]:
df_submit.to_csv("prediction.csv",index=False)